# Self-Supervised Fisheye Rectification Training
## Using NYU Depth V2 Dataset with Kannala-Brandt Parameter Extraction

In [ ]:
# Section 1: Install Dependencies
import subprocess
import sys

packages = ['torch', 'torchvision', 'opencv-python', 'matplotlib', 'numpy', 'tqdm', 'pillow']
for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

In [ ]:
# Section 2: Configuration
import os
from pathlib import Path

CONFIG = {
    'DATASET': {
        'NAME': 'nyu_depth_v2',
        'HEIGHT': 256,
        'WIDTH': 512,
        'DATA_PATH': '/kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_train',
        'MAX_IMAGES': None  # Set to None for all images, or specify number (e.g., 1000)
    },
    'TRAINING': {
        'BATCH_SIZE': 16,
        'NUM_EPOCHS': 50,
        'LEARNING_RATE': 1e-4,
        'NUM_WORKERS': 4,
        'VAL_SPLIT': 0.1,
    },
    'MODEL': {
        'ENCODER': 'vgg11',
        'LATENT_DIM': 512,
    },
    'CHECKPOINT': {
        'PATH': '/kaggle/working/checkpoints/',
        'SAVE_EVERY': 1,
    },
    'OUTPUT': {
        'PATH': '/kaggle/working/outputs/',
    }
}

# Create directories
os.makedirs(CONFIG['CHECKPOINT']['PATH'], exist_ok=True)
os.makedirs(CONFIG['OUTPUT']['PATH'], exist_ok=True)

print(f"Configuration loaded:")
print(f"  Data path: {CONFIG['DATASET']['DATA_PATH']}")
print(f"  Max images: {CONFIG['DATASET']['MAX_IMAGES'] if CONFIG['DATASET']['MAX_IMAGES'] else 'All'}")
print(f"  Batch size: {CONFIG['TRAINING']['BATCH_SIZE']}")
print(f"  Epochs: {CONFIG['TRAINING']['NUM_EPOCHS']}")

In [ ]:
# Section 3: Dataset Preparation
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import random

class NYUFisheyeDataset(Dataset):
    def __init__(self, image_paths, height=256, width=512, augment=False):
        self.image_paths = image_paths
        self.height = height
        self.width = width
        self.augment = augment
        
        self.transform = T.Compose([
            T.Resize((height, width)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
            image = self.transform(image)
            
            # Create synthetic fisheye distortion for training
            distorted, params = self.create_fisheye(image)
            
            return {
                'image': distorted,
                'original': image,
                'params': params
                # Removed 'path' key to fix PosixPath collation error
            }
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return self.__getitem__((idx + 1) % len(self))
    
    def create_fisheye(self, image):
        import cv2
        import numpy as np
        
        img_np = image.permute(1, 2, 0).numpy()
        h, w = img_np.shape[:2]
        
        # Random distortion parameters
        k1 = random.uniform(-0.3, -0.1)
        k2 = random.uniform(0.05, 0.2)
        k3 = random.uniform(-0.01, 0.01)
        k4 = random.uniform(-0.005, 0.005)
        
        cx, cy = w / 2, h / 2
        fx, fy = w / (2 * np.tan(np.pi / 4)), h / (2 * np.tan(np.pi / 4))
        
        # Create distortion map
        y, x = np.mgrid[:h, :w]
        x_norm = (x - cx) / fx
        y_norm = (y - cy) / fy
        r = np.sqrt(x_norm**2 + y_norm**2)
        
        theta = r
        theta_d = theta + k1 * theta**3 + k2 * theta**5 + k3 * theta**7 + k4 * theta**9
        
        scale = np.where(r > 0, theta_d / r, 1)
        map_x = cx + x_norm * scale * fx
        map_y = cy + y_norm * scale * fy
        
        distorted = cv2.remap(img_np, map_x.astype(np.float32), map_y.astype(np.float32), 
                             cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
        
        distorted_tensor = T.ToTensor()(distorted)
        distorted_tensor = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(distorted_tensor)
        
        params = torch.tensor([fx, fy, cx, cy, k1, k2, k3, k4], dtype=torch.float32)
        
        return distorted_tensor, params

# Find all images
data_path = Path(CONFIG['DATASET']['DATA_PATH'])
image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
all_images = []

for ext in image_extensions:
    all_images.extend(list(data_path.rglob(ext)))

print(f"Total images found: {len(all_images)}")

# Limit images if MAX_IMAGES is set
if CONFIG['DATASET']['MAX_IMAGES'] is not None:
    max_imgs = min(CONFIG['DATASET']['MAX_IMAGES'], len(all_images))
    all_images = all_images[:max_imgs]
    print(f"Limited to: {len(all_images)} images")

# Split into train/val
random.shuffle(all_images)
split_idx = int(len(all_images) * (1 - CONFIG['TRAINING']['VAL_SPLIT']))
train_images = all_images[:split_idx]
val_images = all_images[split_idx:]

print(f"Training images: {len(train_images)}")
print(f"Validation images: {len(val_images)}")

train_dataset = NYUFisheyeDataset(train_images, CONFIG['DATASET']['HEIGHT'], CONFIG['DATASET']['WIDTH'])
val_dataset = NYUFisheyeDataset(val_images, CONFIG['DATASET']['HEIGHT'], CONFIG['DATASET']['WIDTH'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['TRAINING']['BATCH_SIZE'], 
                         shuffle=True, num_workers=CONFIG['TRAINING']['NUM_WORKERS'])
val_loader = DataLoader(val_dataset, batch_size=CONFIG['TRAINING']['BATCH_SIZE'], 
                       shuffle=False, num_workers=CONFIG['TRAINING']['NUM_WORKERS'])

In [ ]:
# Section 4: Model Definition
import torch.nn as nn
import torchvision.models as models

class ParametersEstimationModule(nn.Module):
    def __init__(self, encoder_name='vgg11', latent_dim=512):
        super().__init__()
        
        # Encoder
        if encoder_name == 'vgg11':
            vgg = models.vgg11(weights=models.VGG11_Weights.IMAGENET1K_V1)
            self.encoder = vgg.features
            self.avgpool = vgg.avgpool
        
        # Decoder for distortion estimation
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  # Distortion parameter
        )
        
        # Coordinate prediction
        self.coord_predictor = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 2, 3, padding=1)  # x, y coordinates
        )
    
    def forward(self, x):
        features = self.encoder(x)
        pooled = self.avgpool(features)
        flattened = torch.flatten(pooled, 1)
        
        distortion = self.decoder(flattened)
        coordinates = self.coord_predictor(features)
        
        return distortion, coordinates

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

model = ParametersEstimationModule(
    encoder_name=CONFIG['MODEL']['ENCODER'],
    latent_dim=CONFIG['MODEL']['LATENT_DIM']
).to(DEVICE)

In [ ]:
# Section 5: Loss Function (Fixed)
import torch

class DistortionLoss(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, predicted_distortion, coordinate_norms):
        """
        Args:
            predicted_distortion: (B,) tensor of predicted distortion values
            coordinate_norms: (B, N, 2) tensor of coordinate norms
        """
        batch_size = predicted_distortion.shape[0]
        losses = []
        
        for i in range(batch_size):
            pred_dist = predicted_distortion[i]  # Keep as tensor for gradients
            coords = coordinate_norms[i]
            
            # Check if we have enough coordinates
            if coords.numel() == 0 or coords.shape[0] < 2:
                continue
            
            # Get first two coordinates as tensors on same device
            x1 = torch.tensor(float(coords[0][0]), device=predicted_distortion.device)
            y1 = torch.tensor(float(coords[0][1]), device=predicted_distortion.device)
            x2 = torch.tensor(float(coords[1][0]), device=predicted_distortion.device)
            y2 = torch.tensor(float(coords[1][1]), device=predicted_distortion.device)
            
            # Calculate expected distance based on distortion (using torch operations)
            expected_distance = torch.sqrt((x2 - x1)**2 + **(y2 - y1)2) * (1 + pred_dist)
            actual_distance = torch.sqrt((x2 - x1)**2 + **(y2 - y1)2)
            
            loss = torch.abs(expected_distance - actual_distance)
            losses.append(loss)
        
        if len(losses) == 0:
            return torch.tensor(0.0, device=predicted_distortion.device, requires_grad=True)
        
        return torch.stack(losses).mean()

criterion = DistortionLoss()

In [ ]:
# Section 6: Optimizer and Scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['TRAINING']['LEARNING_RATE'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

print(f"Optimizer: Adam")
print(f"Learning rate: {CONFIG['TRAINING']['LEARNING_RATE']}")

In [ ]:
# Section 7: Checkpoint Functions
import glob

def save_checkpoint(epoch, model, optimizer, scheduler, train_losses, val_losses, path):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved: {path}")

def load_checkpoint(path, model, optimizer, scheduler):
    if not os.path.exists(path):
        print(f"No checkpoint found at {path}")
        return 0, [], []
    
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    return start_epoch, train_losses, val_losses

def find_latest_checkpoint(checkpoint_dir):
    checkpoints = glob.glob(os.path.join(checkpoint_dir, 'checkpoint_epoch_*.pth'))
    if not checkpoints:
        return None
    
    latest = max(checkpoints, key=lambda x: int(x.split('_')[-1].split('.')[0]))
    return latest

# Auto-resume from latest checkpoint
latest_checkpoint = find_latest_checkpoint(CONFIG['CHECKPOINT']['PATH'])
start_epoch = 1
train_losses_history = []
val_losses_history = []

if latest_checkpoint:
    print(f"Found existing checkpoint: {latest_checkpoint}")
    start_epoch, train_losses_history, val_losses_history = load_checkpoint(
        latest_checkpoint, model, optimizer, scheduler
    )
    print(f"Resuming from epoch {start_epoch}")
else:
    print("Starting fresh training")

In [ ]:
# Section 8: Training Functions
from tqdm import tqdm

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    
    for batch in tqdm(dataloader, desc="Training"):
        inputs = batch['image'].to(device)
        labels = batch['params'].to(device)
        
        optimizer.zero_grad()
        
        distortion_pred, coords_pred = model(inputs)
        
        # Use distortion parameters as labels for supervision
        loss = criterion(distortion_pred.squeeze(), labels[:, 4])  # k1 parameter
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating"):
            inputs = batch['image'].to(device)
            labels = batch['params'].to(device)
            
            distortion_pred, coords_pred = model(inputs)
            loss = criterion(distortion_pred.squeeze(), labels[:, 4])
            
            total_loss += loss.item()
    
    return total_loss / len(dataloader)

In [ ]:
# Section 9: Training Loop
import matplotlib.pyplot as plt

print(f"\nStarting training from epoch {start_epoch} to {CONFIG['TRAINING']['NUM_EPOCHS']}")
print(f"Device: {DEVICE}")
print(f"Batch size: {CONFIG['TRAINING']['BATCH_SIZE']}")
print(f"Learning rate: {CONFIG['TRAINING']['LEARNING_RATE']}")
print("-" * 60)

for epoch in range(start_epoch, CONFIG['TRAINING']['NUM_EPOCHS'] + 1):
    print(f"\nEpoch {epoch}/{CONFIG['TRAINING']['NUM_EPOCHS']}")
    
    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    
    # Validate
    val_loss = validate_epoch(model, val_loader, criterion, DEVICE)
    
    # Update scheduler
    scheduler.step(val_loss)
    
    # Record losses
    train_losses_history.append(train_loss)
    val_losses_history.append(val_loss)
    
    print(f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
    
    # Save checkpoint
    if epoch % CONFIG['CHECKPOINT']['SAVE_EVERY'] == 0:
        checkpoint_path = os.path.join(
            CONFIG['CHECKPOINT']['PATH'], 
            f"checkpoint_epoch_{epoch}.pth"
        )
        save_checkpoint(epoch, model, optimizer, scheduler, 
                       train_losses_history, val_losses_history, checkpoint_path)
    
    # Plot losses
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses_history, label='Train Loss', marker='o')
    plt.plot(val_losses_history, label='Val Loss', marker='s')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.yscale('log')
    plt.savefig(os.path.join(CONFIG['OUTPUT']['PATH'], 'losses.png'), dpi=150)
    plt.show()

print("\nTraining completed!")

In [ ]:
# Section 10: Inference and Kannala-Brandt Parameter Extraction
import cv2
import numpy as np
import json

def extract_kb_parameters(model, image_path, device):
    """Extract Kannala-Brandt parameters from a trained model"""
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    transform = T.Compose([
        T.Resize((CONFIG['DATASET']['HEIGHT'], CONFIG['DATASET']['WIDTH'])),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        distortion_pred, coords_pred = model(image_tensor)
    
    # Extract KB parameters (estimated from learned distortion)
    h, w = CONFIG['DATASET']['HEIGHT'], CONFIG['DATASET']['WIDTH']
    fx = fy = w / (2 * np.tan(np.pi / 4))
    cx, cy = w / 2, h / 2
    
    k1 = distortion_pred.item()
    k2 = k1 * 0.3  # Estimated ratio
    k3 = k1 * 0.05
    k4 = k1 * 0.01
    
    kb_params = {
        'fx': float(fx),
        'fy': float(fy),
        'cx': float(cx),
        'cy': float(cy),
        'k1': float(k1),
        'k2': float(k2),
        'k3': float(k3),
        'k4': float(k4)
    }
    
    return kb_params, image

def undistort_image(image, kb_params):
    """Undistort image using Kannala-Brandt parameters"""
    img_np = np.array(image)
    h, w = img_np.shape[:2]
    
    fx, fy = kb_params['fx'], kb_params['fy']
    cx, cy = kb_params['cx'], kb_params['cy']
    k1, k2, k3, k4 = kb_params['k1'], kb_params['k2'], kb_params['k3'], kb_params['k4']
    
    # Create undistortion map
    y, x = np.mgrid[:h, :w]
    x_norm = (x - cx) / fx
    y_norm = (y - cy) / fy
    r = np.sqrt(x_norm**2 + y_norm**2)
    
    theta_d = r
    # Inverse distortion (Newton-Raphson approximation)
    theta = theta_d
    for _ in range(5):
        f_theta = theta + k1 * theta**3 + k2 * theta**5 + k3 * theta**7 + k4 * theta**9 - theta_d
        f_prime = 1 + 3*k1*theta**2 + 5*k2*theta**4 + 7*k3*theta**6 + 9*k4*theta**8
        theta = theta - f_theta / f_prime
    
    scale = np.where(r > 0, theta / r, 1)
    map_x = cx + x_norm * scale * fx
    map_y = cy + y_norm * scale * fy
    
    undistorted = cv2.remap(img_np, map_x.astype(np.float32), map_y.astype(np.float32),
                           cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
    
    return undistorted

# Run inference on sample image
if len(all_images) > 0:
    sample_image_path = str(all_images[0])
    print(f"\nRunning inference on: {sample_image_path}")
    
    kb_params, original_img = extract_kb_parameters(model, sample_image_path, DEVICE)
    
    print("\n" + "="*60)
    print("KANNALA-BRANDT PARAMETERS")
    print("="*60)
    print(f"fx: {kb_params['fx']:.2f}")
    print(f"fy: {kb_params['fy']:.2f}")
    print(f"cx: {kb_params['cx']:.2f}")
    print(f"cy: {kb_params['cy']:.2f}")
    print(f"k1: {kb_params['k1']:.6f}")
    print(f"k2: {kb_params['k2']:.6f}")
    print(f"k3: {kb_params['k3']:.6f}")
    print(f"k4: {kb_params['k4']:.6f}")
    print("="*60)
    
    # Undistort
    undistorted_img = undistort_image(original_img, kb_params)
    
    # Plot comparison
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(original_img)
    axes[0].set_title('Original (Distorted)')
    axes[0].axis('off')
    axes[1].imshow(undistorted_img)
    axes[1].set_title('Undistorted')
    axes[1].axis('off')
    plt.tight_layout()
    
    # Save results
    output_path = os.path.join(CONFIG['OUTPUT']['PATH'], 'undistorted_comparison.png')
    plt.savefig(output_path, dpi=150)
    print(f"\nSaved comparison plot: {output_path}")
    
    # Save undistorted image
    undistorted_save_path = os.path.join(CONFIG['OUTPUT']['PATH'], 'undistorted_image.png')
    cv2.imwrite(undistorted_save_path, cv2.cvtColor(undistorted_img, cv2.COLOR_RGB2BGR))
    print(f"Saved undistorted image: {undistorted_save_path}")
    
    # Save KB parameters
    kb_path = os.path.join(CONFIG['OUTPUT']['PATH'], 'kb_parameters.json')
    with open(kb_path, 'w') as f:
        json.dump(kb_params, f, indent=2)
    print(f"Saved KB parameters: {kb_path}")
    
    plt.show()
else:
    print("No images available for inference")